## Install required packages

In [1]:
!pip install geopandas pandas pyarrow

## Import Libraries

In [2]:
import geopandas as gpd
import pandas as pd
import glob

# Marging


## Checking folders

In [6]:
files = glob.glob("/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/*.parquet")
files

['/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N24E092.parquet',
 '/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N25E092.parquet',
 '/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N24E091.parquet',
 '/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N25E091.parquet']

## Read all the tile one by one

In [7]:
gdfs = []

for f in files:
    print("Loading:", f)
    gdf = gpd.read_parquet(f)
    gdfs.append(gdf)

Loading: /content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N24E092.parquet
Loading: /content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N25E092.parquet
Loading: /content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N24E091.parquet
Loading: /content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/Download tile form FTW/2025_N25E091.parquet


## marge all tile

In [8]:
merged_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

## Fix Geometry issues

In [9]:
merged_gdf["geometry"] = merged_gdf["geometry"].buffer(0)

## Remove empty geometry

In [10]:
merged_gdf = merged_gdf[merged_gdf.geometry.notnull()]
merged_gdf = merged_gdf[~merged_gdf.is_empty]
fields = merged_gdf.copy()

# Cliping

## Load Sylhet boundary

In [11]:
sylhet = gpd.read_file("/content/drive/MyDrive/Agri_RS_AI_Project/Projects/02_NDVI_Scale_Sensitivity/01_FTW download and field analysis and sample data extraction/sylhet aoi/sylhet_aoi_shp.shp")

## CRS check and fix

In [12]:
print(fields.crs)
print(sylhet.crs)

{"$schema": "https://proj.org/schemas/v0.5/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal component of 3D system.", "area": "World.", "bb

In [13]:
if fields.crs != sylhet.crs:
    sylhet = sylhet.to_crs(fields.crs)

## Clip

In [14]:
minx, miny, maxx, maxy = sylhet.total_bounds

fields_small = fields.cx[minx:maxx, miny:maxy]

In [15]:
fields_sylhet = gpd.clip(fields_small, sylhet)

## Save

In [16]:
fields_sylhet = fields_sylhet[fields_sylhet.geometry.notnull()]
fields_sylhet = fields_sylhet[~fields_sylhet.is_empty]
fields_sylhet = fields_sylhet.reset_index(drop=True)

fields_sylhet.to_parquet(
    "sylhet_fields.parquet",
    index=False
)